## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到 `solutions/` (这样 `from attention.mha import ...` 这种导入能直接生效)。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd into `solutions/`, turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) != 'solutions':
    if os.path.isdir('solutions'):
        os.chdir('solutions')
    else:
        # already inside a chapter folder — climb out
        while os.path.basename(os.getcwd()) != 'solutions' and os.getcwd() != '/':
            os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the feature_embedding chapter's reference .pt files live
control_folder = 'feature_embedding/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 3 章 · Feature embedding

AF3 的输入端: 把氨基酸 / 原子 / 链 / 模板 / 噪声水平 …… 各种异质特征**翻译**成主干能直接吃的张量。本章先实现两块通用的辅助:

| 文件 | 类 | 作用 |
|---|---|---|
| `relative_position_encoding.py` | `RelativePositionEncoding` | 算法 3 — 每对 token 的相对位置 one-hot |
| `relative_position_encoding.py` | `FourierEmbedding` | 算法 22 — 标量噪声水平的随机 Fourier 嵌入 |

AtomAttentionEncoder / InputFeatureEmbedder 这些重的 wrapper 因为依赖太多上下文，我们这里不做单元测试，等学完 attention/pairformer/diffusion 后在端到端 notebook 里整段验证。

## 3.1 RelativePositionEncoding (算法 3)

打开 `feature_embedding/relative_position_encoding.py`，把 `generate_relp` 和 `forward` 两处 TODO 填好。

`generate_relp` 是关键逻辑: 对每对 token 算 (同链 / 同残基 / 同 entity) 三个布尔门控，再用三组 clip 后的整数偏移做 one-hot。本测试只覆盖最后的线性投影 forward。

In [ ]:
from feature_embedding.relative_position_encoding import RelativePositionEncoding
from feature_embedding.control_values.feature_embedding_checks import (
    r_max, s_max, c_z, test_inputs, test_module_shape, test_module_forward,
)

relpe = RelativePositionEncoding(r_max=r_max, s_max=s_max, c_z=c_z)
test_module_shape(relpe, 'relative_position_encoding', control_folder)
test_module_forward(
    relpe, 'relative_position_encoding',
    inputs=(test_inputs['relp_feature'],),
    output_names='out',
    control_folder=control_folder,
)
print('RelativePositionEncoding ✓')

## 3.2 FourierEmbedding (算法 22)

同一个文件，填 `FourierEmbedding.__init__` 和 `.forward`。

Fourier embedding 把扩散噪声水平这种标量映成 c 维向量: 用一组**固定的**随机系数 w 和 b (在 `__init__` 一次性抽样后作为不可训练参数保存) 算 `cos(2π · (t · w + b))`。

因为 w 和 b 在构造时随机, 要让测试可重复, control values 的 `_generate.py` 调用`torch.manual_seed(0)` 后才生成参数; 你的测试 cell 不需要额外设种子, 保存到 `.pt` 的参数会被替换成 linspace。

In [ ]:
from feature_embedding.relative_position_encoding import FourierEmbedding
from feature_embedding.control_values.feature_embedding_checks import c_noise

fe = FourierEmbedding(c=c_noise)
test_module_shape(fe, 'fourier_embedding', control_folder)
test_module_forward(
    fe, 'fourier_embedding',
    inputs=(test_inputs['noise_level'],),
    output_names='out',
    control_folder=control_folder,
)
print('FourierEmbedding ✓')

## 章节小结

这两块虽小但很关键: RelativePositionEncoding 是 Pairformer 与DiffusionConditioning 共用的 pair 起点; FourierEmbedding 把噪声水平注入 AdaLN 的 single 通道。